In [1]:
import pandas as pd

df = pd.read_csv("Task 3 and 4_Loan_Data.csv")

# default rate across 10 equal-width FICO bands
bands = pd.cut(df["fico_score"], bins=10)
print(df.groupby(bands, observed=True)["default"].mean().round(3))
print()
print("FICO range:", df["fico_score"].min(), "to", df["fico_score"].max())


fico_score
(407.558, 452.2]    0.583
(452.2, 496.4]      0.735
(496.4, 540.6]      0.510
(540.6, 584.8]      0.354
(584.8, 629.0]      0.205
(629.0, 673.2]      0.133
(673.2, 717.4]      0.079
(717.4, 761.6]      0.036
(761.6, 805.8]      0.019
(805.8, 850.0]      0.042
Name: default, dtype: float64

FICO range: 408 to 850


In [2]:
import numpy as np

# For each FICO score: total borrowers and number who defaulted
agg = df.groupby("fico_score")["default"].agg(["count", "sum"]).reset_index()
agg = agg.sort_values("fico_score").reset_index(drop=True)

scores   = agg["fico_score"].values
totals   = agg["count"].values     # borrowers at each score
defaults = agg["sum"].values       # defaulters at each score (sum of 0/1 = count of 1s)

# Prefix sums → any score-range's totals in O(1)
prefix_total   = np.concatenate([[0], np.cumsum(totals)])
prefix_default = np.concatenate([[0], np.cumsum(defaults)])

def bucket_log_likelihood(i, j):
    """Log-likelihood of treating score-groups [i, j) as one bucket."""
    n = prefix_total[j]   - prefix_total[i]      # borrowers in the bucket
    k = prefix_default[j] - prefix_default[i]    # defaulters in the bucket
    if n == 0:
        return 0.0
    p = k / n                                    # the bucket's own default rate = its PD
    if p == 0 or p == 1:
        return 0.0                               # a "pure" bucket is perfectly consistent
    return k * np.log(p) + (n - k) * np.log(1 - p)

print("distinct FICO values:", len(scores))
print("total borrowers:", prefix_total[-1], " total defaults:", prefix_default[-1])


distinct FICO values: 374
total borrowers: 10000  total defaults: 1851


In [3]:
M = len(scores)
K = 5                 # number of buckets

NEG = -1e18
dp    = np.full((K + 1, M + 1), NEG)
split = np.zeros((K + 1, M + 1), dtype=int)
dp[0][0] = 0.0

for b in range(1, K + 1):
    for j in range(1, M + 1):
        for i in range(b - 1, j):                 # last bucket is score-groups [i, j)
            val = dp[b - 1][i] + bucket_log_likelihood(i, j)
            if val > dp[b][j]:
                dp[b][j] = val
                split[b][j] = i

print("max total log-likelihood:", round(dp[K][M], 2))

# Backtrack to recover the boundaries
edges = []
j = M
for b in range(K, 0, -1):
    i = split[b][j]
    edges.append((i, j))
    j = i
edges.reverse()

# Report each bucket
print("\nbucket | FICO range  | borrowers | PD")
for b, (i, j) in enumerate(edges, 1):
    lo, hi = scores[i], scores[j - 1]
    n = prefix_total[j]   - prefix_total[i]
    k = prefix_default[j] - prefix_default[i]
    print(f"  {b}    | {lo:>3}–{hi:<3}    |   {n:>5}   | {k/n:.3f}")

max total log-likelihood: -4255.38

bucket | FICO range  | borrowers | PD
  1    | 408–520    |     301   | 0.661
  2    | 521–580    |    1407   | 0.381
  3    | 581–640    |    3438   | 0.204
  4    | 641–696    |    3197   | 0.105
  5    | 697–850    |    1657   | 0.046


In [4]:
# Upper FICO edge and PD of each bucket
boundaries = [scores[j - 1] for (i, j) in edges]
bucket_pds = []
for (i, j) in edges:
    n = prefix_total[j]   - prefix_total[i]
    k = prefix_default[j] - prefix_default[i]
    bucket_pds.append(k / n)

def fico_to_bucket(score):
    """Map a FICO score to its bucket (1 = lowest FICO / highest risk)."""
    for b, hi in enumerate(boundaries, 1):
        if score <= hi:
            return b
    return len(boundaries)

def bucket_pd(score):
    """Probability of default implied by a borrower's FICO bucket."""
    return round(bucket_pds[fico_to_bucket(score) - 1], 3)

for s in [450, 560, 610, 670, 800]:
    print(f"FICO {s}: bucket {fico_to_bucket(s)}, PD {bucket_pd(s)}")

FICO 450: bucket 1, PD 0.661
FICO 560: bucket 2, PD 0.381
FICO 610: bucket 3, PD 0.204
FICO 670: bucket 4, PD 0.105
FICO 800: bucket 5, PD 0.046
